# Building tool-using agents with LangChain

This notebook introduces LangChain agents by building a small shopping assistant. The assistant can look up product prices and calculate discounts, choosing which tools to call and in what order.

You will learn how to:

1. Define typed tools with the `@tool` decorator.
2. Create an agent with a model, tools, and a system prompt.
3. Invoke the agent and inspect its model and tool messages.
4. Stream intermediate agent steps.
5. Preserve conversation state with a LangGraph checkpointer.

The product catalog is local and deterministic. Only the chat model call requires network access and an OpenAI API key.

## Environment setup

Load `OPENAI_API_KEY` from the project's local `.env` file. The notebook uses `gpt-4.1-mini`, which supports tool calling through LangChain's `ChatOpenAI` integration.

In [1]:
import os
import sys

from dotenv import load_dotenv


load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file."
print(f"Python executable: {sys.executable}")

Python executable: /Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/bin/python3


## Define tools

A tool is a Python function exposed to the model with a name, description, and input schema. LangChain derives the schema from the function's type hints and uses the docstring to tell the model when the tool is useful.

These tools deliberately perform separate jobs. Answering a discounted-price question requires the agent to first look up a product and then pass that result to the calculator.

In [2]:
from langchain.tools import tool


PRODUCT_CATALOG = {
    "laptop stand": 54.99,
    "mechanical keyboard": 89.50,
    "wireless mouse": 34.25,
}


@tool
def get_product_price(product_name: str) -> str:
    """Look up the price of one product in the store catalog."""
    normalized_name = product_name.strip().lower()
    price = PRODUCT_CATALOG.get(normalized_name)
    if price is None:
        available = ", ".join(sorted(PRODUCT_CATALOG))
        return f"Product not found. Available products: {available}."
    return f"{normalized_name}: ${price:.2f}"


@tool
def calculate_discounted_price(price: float, discount_percent: float) -> str:
    """Calculate a final price after applying a percentage discount."""
    if price < 0:
        return "Price must be non-negative."
    if not 0 <= discount_percent <= 100:
        return "Discount percent must be between 0 and 100."

    final_price = price * (1 - discount_percent / 100)
    return f"Final price: ${final_price:.2f}"


tools = [get_product_price, calculate_discounted_price]

/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Inspect the generated tool schemas

The model sees these schemas rather than the Python implementation. Clear names, type hints, parameter names, and docstrings improve tool selection and argument generation.

In [3]:
for current_tool in tools:
    print(f"Tool: {current_tool.name}")
    print(f"Description: {current_tool.description}")
    print(f"Input schema: {current_tool.args}")
    print()

Tool: get_product_price
Description: Look up the price of one product in the store catalog.
Input schema: {'product_name': {'title': 'Product Name', 'type': 'string'}}

Tool: calculate_discounted_price
Description: Calculate a final price after applying a percentage discount.
Input schema: {'price': {'title': 'Price', 'type': 'number'}, 'discount_percent': {'title': 'Discount Percent', 'type': 'number'}}



## Create the agent

`create_agent` builds a model-and-tools loop on top of LangGraph. The model can answer directly or request a tool call. After a tool runs, its result is added to the message history and the model decides whether another tool is needed.

In [4]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

SYSTEM_PROMPT = """You are a precise shopping assistant.
Use get_product_price for every catalog price question.
Wait for the catalog result before requesting any discount calculation.
Then pass the returned numeric price to calculate_discounted_price.
Never request these two tools in parallel or use a placeholder price.
Do not invent prices or calculate discounts yourself.
Briefly explain which product, original price, and discount you used.
"""

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## Invoke the agent

Agent input and output are state dictionaries containing a `messages` sequence. This request should cause two tool calls: one catalog lookup followed by one discount calculation.

In [5]:
request = {
    "messages": [
        {
            "role": "user",
            "content": "What does the mechanical keyboard cost after a 15% discount?",
        }
    ]
}

result = agent.invoke(request)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

The mechanical keyboard originally costs $89.50. After applying a 15% discount, the price is $76.08.


## Inspect the agent trace

The returned state includes the complete message sequence. `AIMessage` objects can contain requested tool calls, and each `ToolMessage` records a tool result linked to the corresponding call ID.

In [6]:
from langchain_core.messages import AIMessage, ToolMessage


for index, message in enumerate(result["messages"], start=1):
    print(f"{index}. {message.type}")

    if isinstance(message, AIMessage) and message.tool_calls:
        for tool_call in message.tool_calls:
            print(f"   requested: {tool_call['name']}({tool_call['args']})")
    elif isinstance(message, ToolMessage):
        print(f"   returned: {message.content}")
    elif message.content:
        print(f"   content: {message.content}")

1. human
   content: What does the mechanical keyboard cost after a 15% discount?
2. ai
   requested: get_product_price({'product_name': 'mechanical keyboard'})
3. tool
   returned: mechanical keyboard: $89.50
4. ai
   requested: calculate_discounted_price({'price': 89.5, 'discount_percent': 15})
5. tool
   returned: Final price: $76.08
6. ai
   content: The mechanical keyboard originally costs $89.50. After applying a 15% discount, the price is $76.08.


## Stream intermediate steps

For longer tasks, streaming exposes progress before the final answer is ready. With `stream_mode="updates"`, each event contains the state update produced by a graph node such as `model` or `tools`.

In [7]:
stream_request = {
    "messages": [
        {
            "role": "user",
            "content": "Find the laptop stand price and apply a 20% discount.",
        }
    ]
}

for update in agent.stream(stream_request, stream_mode="updates"):
    node_name, node_update = next(iter(update.items()))
    latest_message = node_update["messages"][-1]
    print(f"Node: {node_name}")

    if isinstance(latest_message, AIMessage) and latest_message.tool_calls:
        called_tools = [call["name"] for call in latest_message.tool_calls]
        print(f"Tool request: {called_tools}")
    else:
        print(latest_message.content)
    print()

Node: model
Tool request: ['get_product_price']

Node: tools
laptop stand: $54.99



Node: model
Tool request: ['calculate_discounted_price']

Node: tools
Final price: $43.99



Node: model
The original price of the laptop stand is $54.99. After applying a 20% discount, the final price is $43.99.



## Add short-term memory

Agents are stateless unless a checkpointer stores their state. `InMemorySaver` is suitable for local examples and tests. A `thread_id` identifies one conversation; reusing it lets a later invocation access earlier messages.

In production, use a persistent checkpointer rather than in-memory storage.

In [8]:
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver


memory_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid4())}}

first_turn = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My budget is $80. Remember that for my next question.",
            }
        ]
    },
    config=config,
)
first_turn["messages"][-1].pretty_print()

================================== Ai Message ==================================

Got it! Your budget is $80. Let me know what product or discount information you need next.


In [9]:
follow_up = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Would the mechanical keyboard fit my budget after a 15% discount?"
                ),
            }
        ]
    },
    config=config,
)
follow_up["messages"][-1].pretty_print()

================================== Ai Message ==================================

The mechanical keyboard originally costs $89.50. After applying a 15% discount, the price is $76.08, which fits within your $80 budget. Would you like to proceed with this purchase or need information on another product?


## Key takeaways

- Agents combine a chat model, tools, instructions, and an execution loop.
- Tool schemas and descriptions are part of the agent's interface and should be designed carefully.
- The returned message history makes model decisions and tool results inspectable.
- Streaming helps surface progress during multi-step work.
- Checkpointers preserve conversation state when the same `thread_id` is reused.

Good production agents also need tool error handling, authorization boundaries, observability, evaluation, and human approval for consequential actions.